# Intake pipeline: resume to interview plan

Walks one resume through the four intake stages and hands the result to the Interview Agent.

```
resume file -> CandidateResume -> ResumeAnalysis -> InterviewPlan -> InterviewQuestionSet
```

Stages 1-3 each cost one `gpt-5.6` call; stage 4 costs one embedding call per
(competency, difficulty) bucket. Run cells one at a time so a later stage can be
re-run without repaying for the earlier ones.

The equivalent one-shot run is `uv run python scripts/run_intake.py <resume.md>`.

## Setup

`override=True` matters: `python-dotenv` will not replace a variable that is already
exported in your shell, so a stale `PINECONE_INDEX_NAME` silently points retrieval at
the wrong index and every search returns an empty list.

In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=True)

print("project root :", PROJECT_ROOT)
print("index        :", os.getenv("PINECONE_INDEX_NAME"))
print("questions ns :", os.getenv("PINECONE_QUESTION_NAMESPACE"))
print("openai key   :", "set" if os.getenv("OPENAI_API_KEY") else "MISSING")
print("pinecone key :", "set" if os.getenv("PINECONE_API_KEY") else "MISSING")

project root : /home/archana/Documents/AI_ML_GL/GenAcademy/InterviewGapAI
index        : interviewgap-ai
questions ns : questions-v1
openai key   : set
pinecone key : set


## Choose a resume

Point `RESUME_PATH` at any `.md` or `.txt` file. For a PDF, convert it first:
`pdftotext -layout resume.pdf resume.txt`.

In [2]:
RESUME_PATH = PROJECT_ROOT / "data/synthetic/resumes/candidate_03.md"
CANDIDATE_ID = RESUME_PATH.stem

resume_text = RESUME_PATH.read_text(encoding="utf-8").strip()

print(f"{CANDIDATE_ID}  ({len(resume_text)} characters)\n")
print(resume_text[:600] + "...")

candidate_03  (1646 characters)

# Leena Rao

**Intelligent Workflow Engineer**  
leena.rao@example.com

## Professional Summary
Python engineer building assistants that coordinate approved actions across business systems. Four years of experience with service integrations and recoverable task execution.

## Technical Skills
Python, JSON, REST services, PostgreSQL, queues, Docker, service contracts.

## Work Experience
**Intelligent Workflow Engineer — Juniper Relay | Sep 2023–Present**
- Built a service desk assistant that chooses from a limited set of actions, collects missing details, and prepares account-change requests.
...


## Stage 1 — Extract facts

Factual extraction only. No competency judgement happens here, so nothing in this
output should read as an opinion about the candidate.

In [3]:
from src.resume.extractor import extract_resume

resume = extract_resume(resume_text=resume_text, candidate_id=CANDIDATE_ID)

print("name        :", resume.name)
print("target role :", resume.target_role)
print("skills      :", ", ".join(resume.skills))
print("experience  :", len(resume.work_experience), "roles")
print("projects    :", len(resume.projects))

name        : Leena Rao
target role : Intelligent Workflow Engineer
skills      : Python, JSON, REST services, PostgreSQL, queues, Docker, service contracts
experience  : 2 roles
projects    : 1


## Stage 2 — Assess resume evidence

One row per competency. `UNKNOWN_NEEDS_PROBING` means the resume is silent on that
competency, **not** that the candidate is weak in it — which is exactly why it earns a
HIGH probe priority rather than a low score.

In [4]:
from src.resume.analyzer import analyze_resume

analysis = analyze_resume(resume)

print(analysis.overall_summary, "\n")
print(f"{'COMPETENCY':<30}{'EVIDENCE':<24}{'PROBE':<8}CONF")

for item in analysis.competency_evidence:
    print(
        f"{item.competency.value:<30}"
        f"{item.evidence_level.value:<24}"
        f"{item.probe_priority.value:<8}"
        f"{item.confidence:.2f}"
    )

Leena Rao’s resume provides specific evidence of Python-based workflow assistants that coordinate bounded actions across business services, require human approval, validate action arguments, persist execution state, recover from interruptions, and handle repeated failures. It also documents scenario-based testing with simulated service responses and safeguards against duplicate external actions. The resume does not establish retrieval-augmented generation or underlying AI/ML/LLM fundamentals, so those areas should be probed during the interview. 

COMPETENCY                    EVIDENCE                PROBE   CONF
rag                           UNKNOWN_NEEDS_PROBING   HIGH    0.99
agentic_ai                    DEMONSTRATED            LOW     0.98
ai_ml_llm_fundamentals        UNKNOWN_NEEDS_PROBING   HIGH    0.88
ai_evaluation                 DEMONSTRATED            MEDIUM  0.90
python_software_engineering   DEMONSTRATED            LOW     0.98
ai_system_design              DEMONSTRATED  

In [5]:
# Why the analyzer landed where it did, with the resume lines it relied on.
for item in analysis.competency_evidence:
    print(f"\n{item.competency.value.upper()}  ({item.evidence_level.value})")
    print(f"  {item.reason}")
    for evidence in item.evidence:
        print(f"    - {evidence.evidence}")
        print(f"      source: {evidence.source}")


RAG  (UNKNOWN_NEEDS_PROBING)
  The resume does not describe document retrieval, indexing, embeddings, vector search, grounding, reranking, or generation using retrieved context. RAG experience therefore cannot be determined from the supplied evidence.

AGENTIC_AI  (DEMONSTRATED)
  The resume gives clear, specific evidence of assistants selecting bounded actions, gathering missing information, maintaining execution state, recovering from interruption, and handing off after unsuccessful progress.
    - Built a service desk assistant that chooses from a limited set of actions, collects missing details, and prepares account-change requests.
      source: Work Experience - Juniper Relay
    - Saved progress after each completed action so interrupted requests could continue safely.
      source: Work Experience - Juniper Relay
    - Added a maximum action count and a clear handoff when the assistant repeatedly failed to make progress.
      source: Work Experience - Juniper Relay
    - Buil

## Stage 3 — Build the interview plan

The plan is a **retrieval spec**, not questions. Pydantic validators reject it unless the
per-competency counts total 10 and agree with the summary difficulty distribution, so an
arithmetically incoherent plan cannot reach the next stage.

In [6]:
from src.planning.interview_planner import create_interview_plan

plan = create_interview_plan(analysis)

print(f"{'COMPETENCY':<30}{'BAS':>5}{'INT':>5}{'ADV':>5}{'TOTAL':>7}")

for target in plan.competency_targets:
    print(
        f"{target.competency.value:<30}"
        f"{target.basic:>5}{target.intermediate:>5}"
        f"{target.advanced:>5}{target.question_count:>7}"
    )

d = plan.difficulty_distribution
print(f"{'':<30}{d.basic:>5}{d.intermediate:>5}{d.advanced:>5}{plan.total_questions:>7}")

print("\nStrategy:\n ", plan.strategy.rationale)

COMPETENCY                      BAS  INT  ADV  TOTAL
rag                               1    1    0      2
agentic_ai                        0    0    1      1
ai_ml_llm_fundamentals            1    1    0      2
ai_evaluation                     0    1    1      2
python_software_engineering       0    1    0      1
ai_system_design                  0    0    1      1
ai_security                       0    1    0      1
                                  2    5    3     10

Strategy:
  Use a balanced distribution suited to a candidate with substantial Python and workflow-assistant experience. Two basic questions establish foundations in high-priority areas not evidenced by the resume, five intermediate questions test applied understanding across both unknown and demonstrated competencies, and three advanced questions deeply validate the strongest claims in agentic workflows, evaluation, and system design. Resume evidence determines probe emphasis rather than presumed ability, and distin

In [7]:
# The reason behind each allocation. This text is also the retrieval query in stage 4.
for target in plan.competency_targets:
    print(f"\n{target.competency.value} ({target.question_count})")
    print(f"  {target.reason}")


rag (2)
  Allocate two questions because RAG has high probing priority and the resume does not establish retrieval, indexing, grounding, or reranking experience. A basic question checks core concepts, while an intermediate question probes practical design judgment without treating absent evidence as weakness.

agentic_ai (1)
  Allocate one advanced question to validate the strong, specific claims involving bounded actions, durable execution state, interruption recovery, non-progress limits, and human handoff. The depth of the documented professional work supports a focused advanced probe.

ai_ml_llm_fundamentals (2)
  Allocate two questions because this is a high-priority unknown area despite relevant assistant-building context. A basic question establishes conceptual foundations, and an intermediate question tests applied understanding of model behavior and limitations without assuming the implementation used particular models.

ai_evaluation (2)
  Allocate two questions to validate 

## Stage 4 — Resolve the plan to real questions

One filtered Pinecone call per populated bucket, deduplicated across slots, then every
hit joined back to the master corpus for its grading concepts. Check `is_complete`: a
thin difficulty pool produces `unfilled_slots` rather than a silently short interview.

In [8]:
from src.interview.question_selector import select_interview_questions

question_set = select_interview_questions(plan)

print(f"questions : {len(question_set.questions)}")
print(f"complete  : {question_set.is_complete}")

for question in question_set.questions:
    relaxed = "  [difficulty relaxed]" if question.retrieval.filter_relaxed else ""
    print(
        f"\n{question.position:>2}. {question.question_id} "
        f"({question.competency.value} / {question.difficulty}){relaxed}"
    )
    print(f"    {question.question}")
    print(f"    grades on: {'; '.join(question.expected_concepts.must_have)}")

for slot in question_set.unfilled_slots:
    print(f"\nUNFILLED: {slot.reason}")

questions : 10
complete  : True

 1. RAG-CTX-BAS-002 (rag / basic)
    What is the 'lost in the middle' problem in long contexts, and why does it matter for RAG?
    grades on: buried information may be used less effectively; context selection/order affects quality

 2. LLM-TRANS-BAS-001 (ai_ml_llm_fundamentals / basic)
    Using a simple analogy, what roles do Query, Key, and Value play in self-attention?
    grades on: Query represents what a token is looking for; Key represents what other tokens can match against; Value carries information that can be aggregated

 3. RAG-EVAL-INT-002 (rag / intermediate)
    When generating synthetic RAG evaluation questions, what do groundedness, relevance, and stand-alone quality checks verify?
    grades on: groundedness verifies source support; relevance verifies usefulness; stand-alone verifies independent understandability

 4. LLM-TRANS-INT-002 (ai_ml_llm_fundamentals / intermediate)
    Why does a Transformer use multiple attention heads ins

## Hand-off to the Interview Agent

This is the object the agent asks from, one question at a time. `expected_concepts.must_have`
is what the Evaluation Agent later grades against, and what the score divides by.

In [9]:
first = question_set.questions[0]

print(json.dumps(first.model_dump(mode="json"), indent=2))

{
  "position": 1,
  "question_id": "RAG-CTX-BAS-002",
  "competency": "rag",
  "sub_competency": "context_engineering",
  "difficulty": "basic",
  "question_type": "conceptual",
  "question": "What is the 'lost in the middle' problem in long contexts, and why does it matter for RAG?",
  "expected_concepts": {
    "must_have": [
      "buried information may be used less effectively",
      "context selection/order affects quality"
    ],
    "bonus": [
      "reranking",
      "context reduction"
    ]
  },
  "evaluation_refs": [
    "EVAL-RAG-CTX"
  ],
  "plan_reason": "Allocate two questions because RAG has high probing priority and the resume does not establish retrieval, indexing, grounding, or reranking experience. A basic question checks core concepts, while an intermediate question probes practical design judgment without treating absent evidence as weakness.",
  "retrieval": {
    "query": "Allocate two questions because RAG has high probing priority and the resume does not es

In [10]:
# Persist the whole set so the interview session can be replayed without re-planning.
output_dir = PROJECT_ROOT / "data/prepared/interview_questions"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{CANDIDATE_ID}_questions.json"
output_path.write_text(question_set.model_dump_json(indent=2), encoding="utf-8")

print("saved:", output_path.relative_to(PROJECT_ROOT))

saved: data/prepared/interview_questions/candidate_03_questions.json
